# CausalMan: Generate the causal-inference benchmark data

This notebook generates the data required to evaluate the four main causal-inference tasks across **all four CausalMan scales**. For every configured scale and seed it creates:

1. one **observational training dataset**;
2. Task 1 treatment and control distributions;
3. Task 2 treatment and control distributions; and
4. the ATE and product-type-conditioned CATE ground truth reused by Tasks 3 and 4.

The notebook saves raw, observable, mixed-type data. Model-specific ordinal encoding, quantization, one-hot encoding, and normalization are intentionally left to downstream pipelines.

## 1. Protocol encoded here

| Task | Scale(s) | Treatment | Control | Estimand |
|---|---|---|---|---|
| Task 1 | Micro, Small, Medium, Large | `do(PF_M1_T1_Force_LTL = 18000)` | `do(PF_M1_T1_Force_LTL = 15000)` | ATE |
| Task 2 | Micro, Small, Medium, Large | `do(PF_M1_T1_Force = 30000)` | `do(PF_M1_T1_Force = 16000)` | ATE |
| Task 3 | Micro, Small, Medium, Large | Reuses Task 1 arms | Reuses Task 1 arms | CATE by product type |
| Task 4 | Micro, Small, Medium, Large | Reuses Task 2 arms | Reuses Task 2 arms | CATE by product type |

The paper reports 50,000 training samples for Small and 20,000 for Medium, with experimental seeds `4, 6, 42, 66, 90`. It does not report corresponding sample counts for Micro/Large or a separate size for ground-truth treatment/control populations. This notebook uses 50,000 for Micro and 20,000 for Large as explicit neighboring-scale defaults, and uses the same scale-specific size for each interventional arm. All values remain configurable.

Tasks 3–4 condition on product type. The notebook exports CATE for **every product type** and marks 921 as the reference query. Micro contains a single product type, so its product-conditioned CATE is expected to equal its ATE.

## 2. Private repository environment

Run this notebook inside the private CausalMan repository environment. It imports the local `causalman` module and the repository's graph-projection helper. No code is installed, cloned, or fetched automatically.

In [ ]:
import copy
import gc
import hashlib
import importlib.metadata
import json
import math
import os
import platform
import re
import tempfile
import warnings
from datetime import datetime, timezone
from pathlib import Path

import networkx as nx
import numpy as np
import pandas as pd
from IPython.display import display

import causalman
from causalman import CausalMan
from graph_projections import get_latent_projection_single as latent_projection

print(f"Python: {platform.python_version()}")
print(f"CausalMan module: {Path(causalman.__file__).resolve()}")

## 3. Configuration

The default configuration uses all five reported seeds and exports 100 regimes after calibration: four scales times five seeds times one observational plus four interventional regimes. For a quick validation run, set `SEEDS_TO_GENERATE = [42]`.

`batch_multiplier` is not a row-count argument. The notebook calibrates it from an observational run, then uses the same multiplier, seed, and selected row positions for the observational, treated, and control regimes. This keeps batch and product composition aligned across arms.

In [ ]:
OUTPUT_ROOT = Path("output") / "causalman_causal_inference"
ALLOW_OVERWRITE = False
PARALLELIZE = True
DEBUG_MODE = False
SAVE_GROUND_TRUTH_GRAPHS = True
FAIL_ON_MISSING_VALUES = True
STRICT_TASK1_OUTCOME_CHECK = False
REQUIRE_MATCHED_PRODUCT_SEQUENCE = True

PAPER_EXPERIMENT_SEEDS = [4, 6, 42, 66, 90]
SEEDS_TO_GENERATE = PAPER_EXPERIMENT_SEEDS.copy()
# For a quick tutorial run instead, use: SEEDS_TO_GENERATE = [42]

# Small/Medium are paper-reported. Micro/Large are explicit extension defaults.
OBSERVATIONAL_ROWS_BY_SCALE = {
    "micro": 50_000,
    "small": 50_000,
    "medium": 20_000,
    "large": 20_000,
}
# The paper does not state a separate ground-truth arm size.
INTERVENTIONAL_ROWS_PER_ARM_BY_SCALE = {
    "micro": 50_000,
    "small": 50_000,
    "medium": 20_000,
    "large": 20_000,
}
EXPECTED_OBSERVED_GRAPH_NODES = {
    "micro": 33,
    "small": 53,
    "medium": 186,
    "large": 427,
}

INITIAL_BATCH_MULTIPLIER = 1
CALIBRATION_SAFETY_FACTOR = 1.10
MAX_BATCH_MULTIPLIER = 32

OUTCOME_COLUMN_CANDIDATES = ["Sec_C2_Machine1_ProcessResult"]
PRODUCT_TYPE_COLUMN_CANDIDATES = [
    "HU_HU_Block_Type_ID_num",
    "HU_HUBlock_Type_ID_num",
]
REFERENCE_PRODUCT_TYPE_VALUE = 921

print(f"Output root: {OUTPUT_ROOT.resolve()}")

## 4. Declarative task registry

Both arms are atomic interventions. Tasks 3 and 4 do not require additional simulator calls; their CATEs are derived from the corresponding Task 1 and Task 2 arm samples.

In [ ]:
TASKS = [
    {
        "ate_task_id": 1,
        "cate_task_id": 3,
        "slug": "force_ltl",
        "treatment_variable": "PF_M1_T1_Force_LTL",
        "control_value": 15000.0,
        "treatment_value": 18000.0,
        "description": "Raise the lower tolerance limit for the T1 press-fitting force.",
    },
    {
        "ate_task_id": 2,
        "cate_task_id": 4,
        "slug": "force",
        "treatment_variable": "PF_M1_T1_Force",
        "control_value": 16000.0,
        "treatment_value": 30000.0,
        "description": "Raise the T1 press-fitting force to an abnormal value.",
    },
]
SCALES = ["micro", "small", "medium", "large"]

assert [task["ate_task_id"] for task in TASKS] == [1, 2]
assert [task["cate_task_id"] for task in TASKS] == [3, 4]
assert set(SEEDS_TO_GENERATE).issubset(set(PAPER_EXPERIMENT_SEEDS))

plan_rows = []
for scale in SCALES:
    for seed in SEEDS_TO_GENERATE:
        plan_rows.append({
            "scale": scale,
            "seed": seed,
            "observational_rows": OBSERVATIONAL_ROWS_BY_SCALE[scale],
            "rows_per_interventional_arm": INTERVENTIONAL_ROWS_PER_ARM_BY_SCALE[scale],
            "simulator_regimes": 5,
        })
plan_frame = pd.DataFrame(plan_rows)
print(f"Planned simulator regimes after calibration: {len(plan_frame) * 5}")
display(plan_frame)

## 5. Output contract

For each scale and seed, the notebook writes one observational regime and two paired task bundles. Each task bundle contains a treated arm, a control arm, an ATE summary, and CATEs for all product types. Simulator columns that are not DAG nodes are kept in separate row-metadata files and are never added as model covariates.

The observed schema is frozen from the observational DAG and reused across all four intervention regimes. The notebook asserts the paper-reported 53 observed graph nodes for Small and 186 for Medium.

In [ ]:
def package_version(distribution_name):
    try:
        return importlib.metadata.version(distribution_name)
    except importlib.metadata.PackageNotFoundError:
        return "local-or-unversioned"


def canonical_name(name):
    return re.sub(r"[^a-z0-9]", "", str(name).lower())


def resolve_column(columns, candidates, role):
    columns = list(columns)
    exact = [candidate for candidate in candidates if candidate in columns]
    if len(exact) == 1:
        return exact[0]
    canonical_candidates = {canonical_name(candidate) for candidate in candidates}
    canonical_matches = [
        column for column in columns if canonical_name(column) in canonical_candidates
    ]
    matches = list(dict.fromkeys(exact + canonical_matches))
    if len(matches) != 1:
        raise KeyError(
            f"Could not uniquely resolve {role}. Candidates={candidates}, matches={matches}."
        )
    return matches[0]


def observable_flag(node_attributes):
    value = node_attributes.get("Observable", True)
    if isinstance(value, str):
        return value.strip().lower() not in {"false", "0", "no"}
    return bool(value)


def reference_schema(full_data, dag, scale):
    observed_graph_nodes = [
        node for node, attributes in dag.nodes(data=True) if observable_flag(attributes)
    ]
    expected = EXPECTED_OBSERVED_GRAPH_NODES[scale]
    assert len(observed_graph_nodes) == expected, (
        f"{scale}: expected {expected} observed DAG nodes, got {len(observed_graph_nodes)}. "
        "Check the CausalMan revision and Observable attributes."
    )
    missing = [node for node in observed_graph_nodes if node not in full_data.columns]
    assert not missing, f"Observed DAG nodes missing from sampled data: {missing}"
    observed_node_set = set(observed_graph_nodes)
    observed_columns = [
        column for column in full_data.columns if column in observed_node_set
    ]
    assert len(observed_columns) == expected
    non_graph_columns = [column for column in full_data.columns if column not in dag.nodes]
    return observed_columns, non_graph_columns


def validate_observed_frame(frame, outcome_column):
    assert len(frame.columns) == len(set(frame.columns)), "Duplicate columns found."
    missing_values = int(frame.isna().sum().sum())
    if FAIL_ON_MISSING_VALUES:
        assert missing_values == 0, f"Observed data contain {missing_values} missing values."
    numeric = frame.select_dtypes(include=[np.number])
    if numeric.shape[1]:
        assert not np.isinf(numeric.to_numpy(dtype=float)).any(), (
            "Observed data contain positive or negative infinity."
        )
    outcome = pd.to_numeric(frame[outcome_column], errors="raise")
    assert set(outcome.unique()).issubset({0, 1}), (
        f"Outcome {outcome_column} is not binary: {sorted(outcome.unique())}"
    )
    return missing_values


def validate_intervention_table(table, target, expected_rows):
    assert len(table) == expected_rows
    assert target in table.columns, f"Intervention table does not contain {target}."
    positive_columns = set()
    for column in table.columns:
        flags = pd.to_numeric(table[column], errors="raise").to_numpy()
        if np.any(flags != 0):
            positive_columns.add(column)
    assert positive_columns == {target}, (
        f"Unexpected intervened columns: {sorted(positive_columns)}; expected {target}."
    )
    target_flags = pd.to_numeric(table[target], errors="raise").to_numpy()
    assert np.all(target_flags == 1)


def python_scalar(value):
    if isinstance(value, np.generic):
        return value.item()
    return value


def atomic_write_text(text, path):
    path = Path(path)
    temporary_path = path.with_name(path.name + ".tmp")
    temporary_path.write_text(text, encoding="utf-8")
    os.replace(temporary_path, path)


def atomic_write_json(payload, path):
    atomic_write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", path)


def atomic_write_csv(frame, path, compression=None):
    path = Path(path)
    temporary_path = path.with_name(path.name + ".tmp")
    frame.to_csv(temporary_path, index=False, compression=compression)
    os.replace(temporary_path, path)


def graphml_value(value):
    if isinstance(value, np.generic):
        value = value.item()
    if isinstance(value, (str, int, float, bool)):
        return value
    if value is None:
        return "None"
    return str(value)


def atomic_write_graphml(graph, path):
    path = Path(path)
    safe_graph = graph.copy()
    safe_graph.graph.clear()
    for _, attributes in safe_graph.nodes(data=True):
        for key, value in list(attributes.items()):
            attributes[key] = graphml_value(value)
    for _, _, attributes in safe_graph.edges(data=True):
        for key, value in list(attributes.items()):
            attributes[key] = graphml_value(value)
    temporary_path = path.with_name(path.name + ".tmp")
    nx.write_graphml(safe_graph, temporary_path)
    os.replace(temporary_path, path)


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def relative_path(path):
    return Path(path).relative_to(OUTPUT_ROOT).as_posix()


def sample_regime(simulator_name, seed, batch_multiplier, intervention_dict, save_path):
    simulator = CausalMan(
        name=simulator_name,
        seed=seed,
        batch_multiplier=batch_multiplier,
        parallelize=PARALLELIZE,
        debug_mode=DEBUG_MODE,
        save_path=str(save_path),
    )
    if intervention_dict:
        simulator.intervention_dict = intervention_dict
    full_data, intervention_table, simulator_paths, dag = simulator.sample()
    return simulator, full_data, intervention_table, simulator_paths, dag


def calibrate_observational(scale, seed, target_rows, temporary_root):
    simulator_name = f"causalman_{scale}"
    multiplier = INITIAL_BATCH_MULTIPLIER
    attempt = 0
    while True:
        attempt += 1
        save_path = temporary_root / f"{scale}_seed_{seed}_obs_calibration_{attempt}"
        save_path.mkdir(parents=True, exist_ok=True)
        result = sample_regime(simulator_name, seed, multiplier, {}, save_path)
        generated_rows = len(result[1])
        print(
            f"    calibration {scale}, seed={seed}: multiplier={multiplier}, "
            f"rows={generated_rows:,}"
        )
        if generated_rows >= target_rows:
            return multiplier, result
        assert generated_rows > 0
        next_multiplier = max(
            multiplier + 1,
            math.ceil(
                multiplier * target_rows * CALIBRATION_SAFETY_FACTOR / generated_rows
            ),
        )
        del result
        gc.collect()
        if next_multiplier > MAX_BATCH_MULTIPLIER:
            raise RuntimeError(
                f"Required batch multiplier {next_multiplier} exceeds "
                f"MAX_BATCH_MULTIPLIER={MAX_BATCH_MULTIPLIER}."
            )
        multiplier = next_multiplier


def select_common_rows(frame, positions):
    assert int(np.max(positions)) < len(frame)
    return frame.iloc[positions].reset_index(drop=True).copy()


def compute_effects(control, treatment, outcome, evidence, reference_value):
    control_y = pd.to_numeric(control[outcome], errors="raise").astype(float)
    treatment_y = pd.to_numeric(treatment[outcome], errors="raise").astype(float)
    ate = float(treatment_y.mean() - control_y.mean())

    shared_levels = set(control[evidence].dropna().unique()) & set(
        treatment[evidence].dropna().unique()
    )
    cate_rows = []
    for level in sorted(shared_levels, key=lambda value: str(value)):
        control_mask = control[evidence] == level
        treatment_mask = treatment[evidence] == level
        n_control = int(control_mask.sum())
        n_treatment = int(treatment_mask.sum())
        if n_control == 0 or n_treatment == 0:
            continue
        control_mean = float(control_y.loc[control_mask].mean())
        treatment_mean = float(treatment_y.loc[treatment_mask].mean())
        cate_rows.append({
            "product_type_value": python_scalar(level),
            "n_control": n_control,
            "n_treatment": n_treatment,
            "control_outcome_mean": control_mean,
            "treatment_outcome_mean": treatment_mean,
            "cate": treatment_mean - control_mean,
            "is_reference_value": canonical_name(level) == canonical_name(reference_value),
        })

    cate_frame = pd.DataFrame(cate_rows)
    reference_rows = cate_frame.loc[cate_frame["is_reference_value"]]
    assert len(reference_rows) == 1, (
        f"Reference product type {reference_value} is not uniquely represented."
    )
    reference_cate = float(reference_rows.iloc[0]["cate"])
    return {
        "control_outcome_mean": float(control_y.mean()),
        "treatment_outcome_mean": float(treatment_y.mean()),
        "ate": ate,
        "reference_product_type_value": reference_value,
        "reference_cate": reference_cate,
    }, cate_frame

## 6. Generate observational and interventional regimes

Data are generated one regime at a time and full latent samples are released from memory after the observable table has been saved. The same seed, calibrated batch multiplier, and deterministic row positions are used across the five regimes belonging to one scale/seed bundle.

In [ ]:
# Override the sampling helper so CausalMan's private/raw on-disk artifacts are
# deleted immediately after each sample. The returned DataFrames and graph remain in memory.
def sample_regime(simulator_name, seed, batch_multiplier, intervention_dict, save_path):
    save_path = Path(save_path)
    with tempfile.TemporaryDirectory(
        prefix=save_path.name + "_", dir=save_path.parent
    ) as ephemeral_save_path:
        simulator = CausalMan(
            name=simulator_name,
            seed=seed,
            batch_multiplier=batch_multiplier,
            parallelize=PARALLELIZE,
            debug_mode=DEBUG_MODE,
            save_path=ephemeral_save_path,
        )
        if intervention_dict:
            simulator.intervention_dict = intervention_dict
        full_data, intervention_table, simulator_paths, dag = simulator.sample()
    return simulator, full_data, intervention_table, simulator_paths, dag

In [ ]:
def equivalent_category_value(left, right):
    try:
        if float(left) == float(right):
            return True
    except (TypeError, ValueError):
        pass
    return canonical_name(left) == canonical_name(right)


def compute_effects(control, treatment, outcome, evidence, reference_value):
    control_y = pd.to_numeric(control[outcome], errors="raise").astype(float)
    treatment_y = pd.to_numeric(treatment[outcome], errors="raise").astype(float)
    ate = float(treatment_y.mean() - control_y.mean())

    shared_levels = set(control[evidence].dropna().unique()) & set(
        treatment[evidence].dropna().unique()
    )
    cate_rows = []
    for level in sorted(shared_levels, key=lambda value: str(value)):
        control_mask = control[evidence] == level
        treatment_mask = treatment[evidence] == level
        n_control = int(control_mask.sum())
        n_treatment = int(treatment_mask.sum())
        if n_control == 0 or n_treatment == 0:
            continue
        control_mean = float(control_y.loc[control_mask].mean())
        treatment_mean = float(treatment_y.loc[treatment_mask].mean())
        cate_rows.append({
            "product_type_value": python_scalar(level),
            "n_control": n_control,
            "n_treatment": n_treatment,
            "control_outcome_mean": control_mean,
            "treatment_outcome_mean": treatment_mean,
            "cate": treatment_mean - control_mean,
            "is_reference_value": equivalent_category_value(level, reference_value),
        })

    cate_frame = pd.DataFrame(cate_rows)
    reference_rows = cate_frame.loc[cate_frame["is_reference_value"]]
    assert len(reference_rows) == 1, (
        f"Reference product type {reference_value} is not uniquely represented."
    )
    return {
        "control_outcome_mean": float(control_y.mean()),
        "treatment_outcome_mean": float(treatment_y.mean()),
        "ate": ate,
        "reference_product_type_value": reference_value,
        "reference_cate": float(reference_rows.iloc[0]["cate"]),
    }, cate_frame

In [ ]:
if OUTPUT_ROOT.exists() and any(OUTPUT_ROOT.iterdir()) and not ALLOW_OVERWRITE:
    raise FileExistsError(
        f"{OUTPUT_ROOT.resolve()} is not empty. Choose a new OUTPUT_ROOT or "
        "set ALLOW_OVERWRITE=True after verifying its contents."
    )
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

software_versions = {
    "python": platform.python_version(),
    "causalman": getattr(causalman, "__version__", package_version("causalman")),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "networkx": nx.__version__,
}
run_started_utc = datetime.now(timezone.utc).isoformat()
manifest_rows = []
effect_rows = []
graph_saved_for_scale = set()

with tempfile.TemporaryDirectory(prefix="causalman_ci_simulator_") as temporary_root:
    temporary_root = Path(temporary_root)

    for scale in SCALES:
        simulator_name = f"causalman_{scale}"
        observational_target_n = OBSERVATIONAL_ROWS_BY_SCALE[scale]
        interventional_target_n = INTERVENTIONAL_ROWS_PER_ARM_BY_SCALE[scale]
        assert observational_target_n == interventional_target_n, (
            "Common row selection currently requires equal observational and arm sizes."
        )

        for seed in SEEDS_TO_GENERATE:
            print(f"Generating {scale}, seed={seed}")
            bundle_dir = OUTPUT_ROOT / scale / f"seed_{seed:03d}"
            bundle_dir.mkdir(parents=True, exist_ok=ALLOW_OVERWRITE)

            batch_multiplier, observational_result = calibrate_observational(
                scale, seed, observational_target_n, temporary_root
            )
            (
                observational_simulator,
                observational_full,
                observational_intervention_table,
                _observational_paths,
                observational_dag,
            ) = observational_result
            generated_rows = len(observational_full)

            observed_columns, non_graph_columns = reference_schema(
                observational_full, observational_dag, scale
            )
            outcome_column = resolve_column(
                observed_columns, OUTCOME_COLUMN_CANDIDATES, "outcome column"
            )
            product_type_column = resolve_column(
                observed_columns, PRODUCT_TYPE_COLUMN_CANDIDATES, "product-type column"
            )
            resolved_tasks = []
            for task in TASKS:
                resolved_task = copy.deepcopy(task)
                resolved_task["resolved_treatment_variable"] = resolve_column(
                    observed_columns,
                    [task["treatment_variable"]],
                    f"Task {task['ate_task_id']} treatment variable",
                )
                resolved_tasks.append(resolved_task)

            selection_rng = np.random.default_rng(seed + (0 if scale == "small" else 1_000_000))
            row_positions = np.sort(
                selection_rng.choice(generated_rows, size=observational_target_n, replace=False)
            )
            selection_sha256 = hashlib.sha256(row_positions.tobytes()).hexdigest()

            observational_data = select_common_rows(
                observational_full.loc[:, observed_columns], row_positions
            )
            observational_row_metadata = select_common_rows(
                observational_full.loc[:, non_graph_columns], row_positions
            )
            observational_missing = validate_observed_frame(
                observational_data, outcome_column
            )

            observational_dir = bundle_dir / "observational"
            observational_dir.mkdir(parents=True, exist_ok=ALLOW_OVERWRITE)
            observational_data_path = observational_dir / "data_observed.csv.gz"
            observational_metadata_path = observational_dir / "regime.json"
            atomic_write_csv(
                observational_data, observational_data_path, compression="gzip"
            )
            if non_graph_columns:
                atomic_write_csv(
                    observational_row_metadata,
                    observational_dir / "row_metadata.csv.gz",
                    compression="gzip",
                )
            atomic_write_text(
                "\n".join(observed_columns) + "\n",
                bundle_dir / "observable_variables.txt",
            )
            atomic_write_json(
                {column: str(dtype) for column, dtype in observational_data.dtypes.items()},
                bundle_dir / "data_schema.json",
            )

            if SAVE_GROUND_TRUTH_GRAPHS and scale not in graph_saved_for_scale:
                graph_dir = OUTPUT_ROOT / scale / "ground_truth_graphs"
                graph_dir.mkdir(parents=True, exist_ok=True)
                projected_admg = latent_projection(observational_dag.copy())
                assert projected_admg.number_of_nodes() == EXPECTED_OBSERVED_GRAPH_NODES[scale]
                atomic_write_graphml(
                    observational_dag, graph_dir / "ground_truth_dag.graphml"
                )
                atomic_write_graphml(
                    projected_admg, graph_dir / "ground_truth_admg.graphml"
                )
                graph_saved_for_scale.add(scale)

            observational_sha256 = sha256_file(observational_data_path)
            observational_metadata = {
                "schema_version": "1.0",
                "role": "train_observational",
                "scale": scale,
                "simulator_name": simulator_name,
                "seed": seed,
                "batch_multiplier": batch_multiplier,
                "generated_rows": generated_rows,
                "exported_rows": observational_target_n,
                "observed_columns": len(observed_columns),
                "non_graph_columns": non_graph_columns,
                "outcome_column": outcome_column,
                "product_type_column": product_type_column,
                "missing_values": observational_missing,
                "row_selection_sha256": selection_sha256,
                "data_sha256": observational_sha256,
                "software_versions": software_versions,
                "generated_utc": datetime.now(timezone.utc).isoformat(),
            }
            atomic_write_json(observational_metadata, observational_metadata_path)
            manifest_rows.append({
                "role": "train_observational",
                "scale": scale,
                "seed": seed,
                "ate_task_id": None,
                "cate_task_id": None,
                "arm": "observational",
                "treatment_variable": None,
                "intervention_value": None,
                "outcome_column": outcome_column,
                "product_type_column": product_type_column,
                "reference_product_type_value": REFERENCE_PRODUCT_TYPE_VALUE,
                "batch_multiplier": batch_multiplier,
                "generated_rows": generated_rows,
                "exported_rows": observational_target_n,
                "observed_columns": len(observed_columns),
                "data_path": relative_path(observational_data_path),
                "metadata_path": relative_path(observational_metadata_path),
                "data_sha256": observational_sha256,
            })

            observational_product_sequence = observational_data[
                product_type_column
            ].reset_index(drop=True)

            for task in resolved_tasks:
                ate_task_id = task["ate_task_id"]
                cate_task_id = task["cate_task_id"]
                target = task["resolved_treatment_variable"]
                task_dir = bundle_dir / (
                    f"task_{ate_task_id}_ate_task_{cate_task_id}_cate_{task['slug']}"
                )
                task_dir.mkdir(parents=True, exist_ok=ALLOW_OVERWRITE)
                arm_data = {}

                for arm, value in [
                    ("control", task["control_value"]),
                    ("treatment", task["treatment_value"]),
                ]:
                    print(
                        f"    Task {ate_task_id}, {arm}: do({target}={value:g}), "
                        f"multiplier={batch_multiplier}"
                    )
                    simulator_save_path = temporary_root / (
                        f"{scale}_seed_{seed}_task_{ate_task_id}_{arm}"
                    )
                    simulator_save_path.mkdir(parents=True, exist_ok=True)
                    (
                        arm_simulator,
                        arm_full,
                        arm_intervention_table,
                        _arm_paths,
                        arm_dag,
                    ) = sample_regime(
                        simulator_name,
                        seed,
                        batch_multiplier,
                        {target: value},
                        simulator_save_path,
                    )
                    assert len(arm_full) == generated_rows, (
                        "Matched row selection requires equal raw row counts across regimes."
                    )
                    missing_columns = [
                        column for column in observed_columns if column not in arm_full.columns
                    ]
                    assert not missing_columns, f"Arm is missing observed columns: {missing_columns}"

                    selected_arm_data = select_common_rows(
                        arm_full.loc[:, observed_columns], row_positions
                    )
                    selected_arm_metadata = select_common_rows(
                        arm_full.loc[:, non_graph_columns], row_positions
                    )
                    selected_arm_mask = select_common_rows(
                        arm_intervention_table, row_positions
                    )
                    assert len(selected_arm_data) == interventional_target_n
                    assert np.allclose(
                        pd.to_numeric(selected_arm_data[target], errors="raise"),
                        value,
                        rtol=0.0,
                        atol=1e-10,
                    )
                    validate_intervention_table(
                        selected_arm_mask, target, interventional_target_n
                    )
                    arm_missing = validate_observed_frame(
                        selected_arm_data, outcome_column
                    )

                    if REQUIRE_MATCHED_PRODUCT_SEQUENCE:
                        assert selected_arm_data[product_type_column].reset_index(
                            drop=True
                        ).equals(observational_product_sequence), (
                            "Product-type sequence differs across matched regimes."
                        )

                    arm_dir = task_dir / arm
                    arm_dir.mkdir(parents=True, exist_ok=ALLOW_OVERWRITE)
                    arm_data_path = arm_dir / "data_observed.csv.gz"
                    arm_mask_path = arm_dir / "intervention_table.csv.gz"
                    arm_metadata_path = arm_dir / "regime.json"
                    atomic_write_csv(
                        selected_arm_data, arm_data_path, compression="gzip"
                    )
                    atomic_write_csv(
                        selected_arm_mask, arm_mask_path, compression="gzip"
                    )
                    if non_graph_columns:
                        atomic_write_csv(
                            selected_arm_metadata,
                            arm_dir / "row_metadata.csv.gz",
                            compression="gzip",
                        )
                    arm_sha256 = sha256_file(arm_data_path)
                    arm_metadata_payload = {
                        "schema_version": "1.0",
                        "role": "ground_truth_interventional",
                        "scale": scale,
                        "simulator_name": simulator_name,
                        "seed": seed,
                        "ate_task_id": ate_task_id,
                        "cate_task_id": cate_task_id,
                        "arm": arm,
                        "intervention_type": "atomic",
                        "treatment_variable": target,
                        "intervention_value": value,
                        "outcome_column": outcome_column,
                        "product_type_column": product_type_column,
                        "reference_product_type_value": REFERENCE_PRODUCT_TYPE_VALUE,
                        "batch_multiplier": batch_multiplier,
                        "generated_rows": len(arm_full),
                        "exported_rows": interventional_target_n,
                        "observed_columns": len(observed_columns),
                        "missing_values": arm_missing,
                        "row_selection_sha256": selection_sha256,
                        "data_sha256": arm_sha256,
                        "software_versions": software_versions,
                        "generated_utc": datetime.now(timezone.utc).isoformat(),
                    }
                    atomic_write_json(arm_metadata_payload, arm_metadata_path)
                    manifest_rows.append({
                        "role": "ground_truth_interventional",
                        "scale": scale,
                        "seed": seed,
                        "ate_task_id": ate_task_id,
                        "cate_task_id": cate_task_id,
                        "arm": arm,
                        "treatment_variable": target,
                        "intervention_value": value,
                        "outcome_column": outcome_column,
                        "product_type_column": product_type_column,
                        "reference_product_type_value": REFERENCE_PRODUCT_TYPE_VALUE,
                        "batch_multiplier": batch_multiplier,
                        "generated_rows": len(arm_full),
                        "exported_rows": interventional_target_n,
                        "observed_columns": len(observed_columns),
                        "data_path": relative_path(arm_data_path),
                        "metadata_path": relative_path(arm_metadata_path),
                        "data_sha256": arm_sha256,
                    })
                    arm_data[arm] = selected_arm_data

                    del arm_simulator, arm_full, selected_arm_metadata
                    del arm_intervention_table, selected_arm_mask, arm_dag
                    gc.collect()

                if REQUIRE_MATCHED_PRODUCT_SEQUENCE:
                    assert arm_data["control"][product_type_column].reset_index(
                        drop=True
                    ).equals(
                        arm_data["treatment"][product_type_column].reset_index(drop=True)
                    )

                effect_summary, cate_frame = compute_effects(
                    arm_data["control"],
                    arm_data["treatment"],
                    outcome_column,
                    product_type_column,
                    REFERENCE_PRODUCT_TYPE_VALUE,
                )
                if ate_task_id == 1 and effect_summary["treatment_outcome_mean"] != 0.0:
                    message = (
                        "Task 1 treatment did not make every ProcessResult zero: "
                        f"mean={effect_summary['treatment_outcome_mean']:.6g}."
                    )
                    if STRICT_TASK1_OUTCOME_CHECK:
                        raise AssertionError(message)
                    warnings.warn(message)

                cate_frame.insert(0, "scale", scale)
                cate_frame.insert(1, "seed", seed)
                cate_frame.insert(2, "cate_task_id", cate_task_id)
                cate_path = task_dir / "cate_by_product_type.csv"
                effects_path = task_dir / "effects.json"
                atomic_write_csv(cate_frame, cate_path)
                atomic_write_json(
                    {
                        "schema_version": "1.0",
                        "scale": scale,
                        "seed": seed,
                        "ate_task_id": ate_task_id,
                        "cate_task_id": cate_task_id,
                        "treatment_variable": target,
                        "control_value": task["control_value"],
                        "treatment_value": task["treatment_value"],
                        "outcome_column": outcome_column,
                        "product_type_column": product_type_column,
                        **effect_summary,
                    },
                    effects_path,
                )
                effect_rows.append({
                    "scale": scale,
                    "seed": seed,
                    "ate_task_id": ate_task_id,
                    "cate_task_id": cate_task_id,
                    "treatment_variable": target,
                    "control_value": task["control_value"],
                    "treatment_value": task["treatment_value"],
                    "outcome_column": outcome_column,
                    "product_type_column": product_type_column,
                    **effect_summary,
                    "cate_table_path": relative_path(cate_path),
                    "effects_path": relative_path(effects_path),
                })
                print(
                    f"    Task {ate_task_id}: ATE={effect_summary['ate']:.6g}, "
                    f"CATE(z={REFERENCE_PRODUCT_TYPE_VALUE})="
                    f"{effect_summary['reference_cate']:.6g}"
                )
                del arm_data, cate_frame
                gc.collect()

            del observational_simulator, observational_full
            del observational_intervention_table, observational_dag
            del observational_data, observational_row_metadata
            gc.collect()

manifest_frame = pd.DataFrame(manifest_rows)
effects_frame = pd.DataFrame(effect_rows)
expected_bundles = len(SCALES) * len(SEEDS_TO_GENERATE)
assert len(manifest_frame) == expected_bundles * 5
assert len(effects_frame) == expected_bundles * 2

atomic_write_csv(manifest_frame, OUTPUT_ROOT / "manifest.csv")
atomic_write_csv(effects_frame, OUTPUT_ROOT / "effects.csv")
atomic_write_json(manifest_rows, OUTPUT_ROOT / "manifest.json")
atomic_write_json(effect_rows, OUTPUT_ROOT / "effects.json")
atomic_write_json(
    {
        "schema_version": "1.0",
        "run_started_utc": run_started_utc,
        "run_completed_utc": datetime.now(timezone.utc).isoformat(),
        "scales": SCALES,
        "seeds": SEEDS_TO_GENERATE,
        "paper_experiment_seeds": PAPER_EXPERIMENT_SEEDS,
        "observational_rows_by_scale": OBSERVATIONAL_ROWS_BY_SCALE,
        "interventional_rows_per_arm_by_scale": (
            INTERVENTIONAL_ROWS_PER_ARM_BY_SCALE
        ),
        "reference_product_type_value": REFERENCE_PRODUCT_TYPE_VALUE,
        "software_versions": software_versions,
    },
    OUTPUT_ROOT / "run_config.json",
)

In [ ]:
# Final release checks
expected_bundles = len(SCALES) * len(SEEDS_TO_GENERATE)
expected_regimes = expected_bundles * 5
expected_effects = expected_bundles * 2
assert len(manifest_frame) == expected_regimes
assert len(effects_frame) == expected_effects
assert not manifest_frame.duplicated(
    subset=["scale", "seed", "ate_task_id", "arm"], keep=False
).any()

for row in manifest_rows:
    data_path = OUTPUT_ROOT / row["data_path"]
    metadata_path = OUTPUT_ROOT / row["metadata_path"]
    assert data_path.is_file() and data_path.stat().st_size > 0
    assert metadata_path.is_file() and metadata_path.stat().st_size > 0
    assert sha256_file(data_path) == row["data_sha256"]
    reloaded = pd.read_csv(data_path)
    assert len(reloaded) == row["exported_rows"]
    assert len(reloaded.columns) == row["observed_columns"]

print(
    f"Successfully generated {len(manifest_frame)} regimes and "
    f"{len(effects_frame)} effect summaries in {OUTPUT_ROOT.resolve()}"
)
display(manifest_frame)
display(effects_frame)

## 7. Interpretation and design choices

- `manifest.csv` contains one row per generated regime: one observational dataset plus four interventional arms for every scale/seed.
- `effects.csv` contains the ATE and reference CATE for Tasks 1–4. Each task directory additionally contains `cate_by_product_type.csv` for all observed product types.
- Both control groups are generated interventionally. The missing `do` on the control term in the paper's generic ATE equation is treated as a typesetting error because the task equations and surrounding text explicitly define control interventions.
- The paper-reported 50,000/20,000 values most clearly describe Small/Medium model-training sizes. Micro=50,000 and Large=20,000 are extension choices, and using the same sizes for each ground-truth arm is also a documented choice because a separate Monte Carlo arm size is not stated.
- The exact product-type column spelling is resolved from two known candidates and checked against runtime columns. The benchmark query defaults to product type 921, while all product-level CATEs are retained. With one product type, Micro's CATE should coincide with its ATE.
- The notebook uses raw mixed-type data for ground-truth distributions and effects. Any ordinal encoding, min-max scaling, CBN quantization, or neural-model-specific transformation must be fit on the observational training data and applied consistently downstream.

### Directory sketch

```text
output/causalman_causal_inference/
├── manifest.csv
├── effects.csv
├── run_config.json
├── micro/ ...
├── small/
│   ├── ground_truth_graphs/
│   │   ├── ground_truth_dag.graphml
│   │   └── ground_truth_admg.graphml
│   └── seed_004/
│       ├── observational/data_observed.csv.gz
│       ├── task_1_ate_task_3_cate_force_ltl/
│       │   ├── control/data_observed.csv.gz
│       │   ├── treatment/data_observed.csv.gz
│       │   ├── effects.json
│       │   └── cate_by_product_type.csv
│       └── task_2_ate_task_4_cate_force/ ...
├── medium/ ...
└── large/ ...
```